In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] load the llm_math package text: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 13. text — text Vector Spacetext

> **Learning Goals**
> - Word2Vec(Skip-gram, CBOW)text text Functiontext textdegreestext
> - GloVetext text Matrix text text
> - text Training text text text text

## 13.1 text text text

"text text text text text text text" (Firth, 1957).

**text text(distributed representation)**: text textDimension Vectortext text Dimensiontext text Encoding.
- "king" = [0.9, 0.7, 0.1, ...] (text, text, ...)

## 13.2 Word2Vec — Skip-gram

text text text text text:
$$\max_\theta \sum_{t=1}^{T} \sum_{-c \leq j \leq c, j \neq 0} \log P(w_{t+j} | w_t; \theta)$$

text Model:
$$P(w_O | w_I) = \frac{\exp(\mathbf{v}'_{w_O}{}^\top \mathbf{v}_{w_I})}{\sum_{w \in V} \exp(\mathbf{v}'_w{}^\top \mathbf{v}_{w_I})}$$

- $\mathbf{v}_w$: text $w$text text Vector
- $\mathbf{v}'_w$: text $w$text Output Vector
- text Calculationtext $O(|V|)$text text → **Negative Sampling** text


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Skip-gram with Negative Sampling (text)
class SkipGramNS:
    def __init__(self, vocab_size, embed_dim=50, seed=0):
        rng = np.random.default_rng(seed)
        self.W_in = rng.standard_normal((vocab_size, embed_dim)) * 0.01
        self.W_out = rng.standard_normal((vocab_size, embed_dim)) * 0.01
        self.embed_dim = embed_dim

    def forward(self, center_idx, context_idx, neg_indices):
        v = self.W_in[center_idx]      # (D,)
        u_pos = self.W_out[context_idx]  # (D,)
        u_neg = self.W_out[neg_indices]  # (K, D)
        return v, u_pos, u_neg

    def loss_and_grad(self, center_idx, context_idx, neg_indices):
        v = self.W_in[center_idx]
        u_pos = self.W_out[context_idx]
        u_neg = self.W_out[neg_indices]  # (K, D)

        # positive: log sigma(u_pos . v)
        score_pos = u_pos @ v
        sigma_pos = 1 / (1 + np.exp(-score_pos))
        loss = -np.log(sigma_pos + 1e-12)

        # negative: log sigma(-u_neg . v)
        score_neg = u_neg @ v  # (K,)
        sigma_neg = 1 / (1 + np.exp(-score_neg))
        loss -= np.sum(np.log(1 - sigma_neg + 1e-12))

        # text
        dloss_dscore_pos = sigma_pos - 1
        dloss_dscore_neg = sigma_neg  # (K,)

        grad_v = dloss_dscore_pos * u_pos + np.sum(dloss_dscore_neg[:, None] * u_neg, axis=0)
        grad_u_pos = dloss_dscore_pos * v
        grad_u_neg = dloss_dscore_neg[:, None] * v[None, :]

        return loss, grad_v, grad_u_pos, grad_u_neg

    def train_step(self, center_idx, context_idx, neg_indices, lr=0.025):
        loss, gv, gu_pos, gu_neg = self.loss_and_grad(center_idx, context_idx, neg_indices)
        self.W_in[center_idx] -= lr * gv
        self.W_out[context_idx] -= lr * gu_pos
        self.W_out[neg_indices] -= lr * gu_neg
        return loss

# text text
corpus_text = "the king killed the queen and the prince was sad and the kingdom fell"
words = corpus_text.split()
vocab = sorted(set(words))
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}
vocab_size = len(vocab)
print(f"Vocabulary Size: {vocab_size}, text Length: {len(words)}")

# Training Data text (skip-gram pairs)
window = 2
pairs = []
for i, w in enumerate(words):
    for j in range(max(0, i - window), min(len(words), i + window + 1)):
        if j != i:
            pairs.append((word_to_idx[w], word_to_idx[words[j]]))
print(f"Training text text: {len(pairs)}")

# text text Distribution (unigram^0.75)
word_freqs = Counter(words)
neg_dist = np.array([word_freqs[w]**0.75 for w in vocab])
neg_dist /= neg_dist.sum()

# Training
model = SkipGramNS(vocab_size, embed_dim=20, seed=0)
losses = []
for epoch in range(50):
    total_loss = 0
    np.random.shuffle(pairs)
    for center, context in pairs:
        K = 5
        negs = np.random.choice(vocab_size, size=K, p=neg_dist)
        total_loss += model.train_step(center, context, negs, lr=0.05)
    losses.append(total_loss / len(pairs))

plt.figure(figsize=(9, 4))
plt.plot(losses, 'b-')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Skip-gram with Negative Sampling Learning Curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch13_word2vec_loss.png', dpi=100, bbox_inches='tight')
plt.show()


## 13.3 text text (Word Analogy)

Word2Vectext text Result:
$$\mathbf{v}(\text{king}) - \mathbf{v}(\text{man}) + \mathbf{v}(\text{woman}) \approx \mathbf{v}(\text{queen})$$

Vector Spacetext text, text text text Dimensiontext Trainingtext.


In [ ]:
# text textdegrees Calculation (text Model)
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)

# Trainingtext text textdegrees
print("Trainingtext word Embedding Similarity:")
for w1 in ['king', 'queen', 'kingdom']:
    sims = []
    for w2 in vocab:
        if w1 != w2:
            v1 = model.W_in[word_to_idx[w1]]
            v2 = model.W_in[word_to_idx[w2]]
            sims.append((w2, cosine_sim(v1, v2)))
    sims.sort(key=lambda x: -x[1])
    print(f"  '{w1}'text text text word: {sims[:3]}")


## 13.4 GloVe — text text text

GloVe (Pennington et al., 2014)text text text text(co-occurrence) text text Training.

$$J = \sum_{i,j} f(X_{ij}) \left(\mathbf{w}_i^\top \tilde{\mathbf{w}}_j + b_i + \tilde{b}_j - \log X_{ij}\right)^2$$

- $X_{ij}$: text $i$text $j$text text text
- $f(X_{ij})$: text Function (text text text text text text)
- $\mathbf{w}, \tilde{\mathbf{w}}$: text text text Vector (text text)

Word2Vectext text text text text. text text text text.


In [ ]:
# text Matrix Calculation (text)
def build_cooc_matrix(words, window=2, vocab_size=None, word_to_idx=None):
    """word text Matrix."""
    if vocab_size is None:
        vocab = sorted(set(words))
        word_to_idx = {w: i for i, w in enumerate(vocab)}
        vocab_size = len(vocab)
    M = np.zeros((vocab_size, vocab_size))
    for i, w in enumerate(words):
        wi = word_to_idx[w]
        for j in range(max(0, i - window), min(len(words), i + window + 1)):
            if j != i:
                wj = word_to_idx[words[j]]
                # Distancetext text Weight
                weight = 1.0 / abs(i - j)
                M[wi, wj] += weight
    return M, word_to_idx

M, w2i = build_cooc_matrix(words, window=2, vocab_size=vocab_size, word_to_idx=word_to_idx)
print(f"text Matrix: {M.shape}")
print(f"text 5x5:")
print(M[:5, :5].round(2))
print(f"\n'king'text text text text word:")
king_idx = word_to_idx['king']
cooc = [(idx_to_word[i], M[king_idx, i]) for i in range(vocab_size) if i != king_idx]
cooc.sort(key=lambda x: -x[1])
for w, c in cooc[:5]:
    print(f"  {w}: {c:.2f}")


## 13.5 text Training text text text

Word2Vec/GloVetext **text text** — text text text text Vector.

Problem: "bank" (text)text "bank" (text)text text Vector → **text Problem**.

text **Training text text Matrix** $\mathbf{E} \in \mathbb{R}^{|V| \times d}$:
- text ID $i$ → $\mathbf{E}[i]$
- text text Attentiontext text text text text
- "bank"text text text text, Attention text text text text

text **text text**text textdegreestext text (Positional Encoding, Ch 16text text).

text text: text text text $\sqrt{d}$text text.
$$\mathbf{E}' = \mathbf{E} \cdot \sqrt{d}$$


In [ ]:
# text text (PyTorch)
import torch
import torch.nn as nn

vocab_size = 1000
embed_dim = 64

# Training text text
embedding = nn.Embedding(vocab_size, embed_dim)
print(f"Embedding Matrix: {embedding.weight.shape}")
print(f"Parameter Count: {embedding.weight.numel():,}")

# text
seq = torch.tensor([[1, 5, 10, 25, 100]])  # (batch, seq_len)
emb = embedding(seq)  # (batch, seq_len, embed_dim)
emb_scaled = emb * (embed_dim ** 0.5)

print(f"\nEmbedding Output: {emb.shape}")
print(f"Scaling text std: {emb.std():.4f}")
print(f"Scaling text std: {emb_scaled.std():.4f} (≈ {embed_dim ** 0.5:.2f}text)")

# Visualization: text Vector (text 20Dimension)
fig, ax = plt.subplots(figsize=(12, 4))
emb_sample = embedding.weight[:20, :20].detach().numpy()
im = ax.imshow(emb_sample, cmap='RdBu', aspect='auto')
plt.colorbar(im, ax=ax)
ax.set_xlabel('Embedding Dimension')
ax.set_ylabel('text ID')
ax.set_title('Embedding Matrix (text 20x20)')
plt.tight_layout()
plt.savefig('../figures/ch13_embedding_matrix.png', dpi=100, bbox_inches='tight')
plt.show()


## 13.6 text Visualization — t-SNE, PCA

textDimension text 2Dtext text Visualization. text text text text.


In [ ]:
# text Visualization (text PCA)
from sklearn.decomposition import PCA

# Trainingtext text text
embeddings = model.W_in  # (vocab_size, embed_dim)
print(f"Embedding: {embeddings.shape}")

# PCAtext 2D text
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(embeddings)

# Visualization
fig, ax = plt.subplots(figsize=(8, 7))
for i, word in enumerate(vocab):
    ax.scatter(emb_2d[i, 0], emb_2d[i, 1], c='blue', s=80)
    ax.annotate(word, (emb_2d[i, 0], emb_2d[i, 1]), fontsize=11)
ax.set_title('Word2Vec Embedding (PCA 2D text)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch13_embedding_tsne.png', dpi=100, bbox_inches='tight')
plt.show()
print("text wordtext text text text (text text Patterntext text text text).")


## 13.7 Key Takeaways

| Model | text | text |
|---|---|---|
| Word2Vec | text text (skip-gram/CBOW) | text text |
| GloVe | text text | text text text |
| FastText | text Word2Vec | OOV text text |
| Transformer text | Training text Matrix | text text text |

## Exercises

1. Skip-gramtext CBOWtext text text, text text text text text.
2. text text text $K = 1, 5, 15$text text Training Resulttext Comparisontext.
3. GloVetext text Function $f(x) = (x/x_{\max})^\alpha$ if $x < x_{\max}$ else 1 text text text.
4. Word2Vectext text text text text text Resulttext text.
5. text text $\sqrt{d}$ text text text Attention text text text.

> Solutions: `solutions/ch13_solutions.ipynb`
